<a href="https://colab.research.google.com/github/NaziaUrmi/Customer_Payments_Analytics/blob/main/notebooks/Project_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import files

uploaded = files.upload()

Saving customer-payments-analytics.zip to customer-payments-analytics.zip


upload zip file

In [12]:
import zipfile
from pathlib import Path

zip_path = "/content/customer-payments-analytics.zip"
extract_path = "/content/New-Skills"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

project = Path(extract_path) / "customer-payments-analytics"

print("Project folder:", project)
print("Project files:")
for path in sorted(project.rglob("*")):
    if path.is_file():
        print(path.relative_to(project))

Project folder: /content/New-Skills/customer-payments-analytics
Project files:
README.md
data/processed/customers_review.csv
data/processed/data_quality_report.csv
data/processed/invoices_review.csv
data/processed/payments_review.csv
data/raw/customers.csv
data/raw/data_quality_issues.csv
data/raw/invoices.csv
data/raw/payments.csv
excel/README.md
notebooks/phase1_exercises.md
powerbi/README.md
reports/data_dictionary.md
reports/phase1_generation_log.txt
requirements.txt
sql/01_create_schema.sql
sql/02_load_csv_guidance.sql
sql/03_starter_queries.sql
src/clean_data.py
src/generate_data.py
tests/__pycache__/test_data_quality.cpython-313.pyc
tests/test_data_quality.py


Extract zip file


In [13]:
%pip install -q pandas numpy matplotlib

Install the required Python libraries

In [14]:
import pandas as pd

customers = pd.read_csv(project / "data/raw/customers.csv")

print("Dataset shape:", customers.shape)
display(customers.head())

Dataset shape: (1001, 6)


,customer_id,customer_name,segment,region,customer_since,payment_terms_days
0,C00001,Customer 0001,Small Business,DACH,2020-04-04,30
1,C00002,Customer 0002,Small Business,DACH,2021-05-16,14
2,C00003,Customer 0003,Small Business,DACH,2021-06-22,30
3,C00004,Customer 0004,Small Business,DACH,2021-09-13,30
4,C00005,Customer 0005,Small Business,DACH,2024-07-13,30


oad the customer dataset, show the first five customer records

In [15]:
# Count duplicate customer IDs
print(
    "Duplicate customer IDs:",
    customers["customer_id"].duplicated().sum()
)

# Count missing values in each column
print("\nMissing values:")
print(customers.isna().sum())

Duplicate customer IDs: 1

Missing values:
customer_id           0
customer_name         0
segment               1
region                0
customer_since        0
payment_terms_days    0
dtype: int64


check for duplicate customer IDs and missing values

### Group and count customers by segment

In [18]:
# Group by segment and count the number of customers
segment_counts = customers['segment'].value_counts(dropna=False).reset_index()
segment_counts.columns = ['Segment', 'Customer Count']

# Display the counts
display(segment_counts)

,Segment,Customer Count
0,Small Business,519
1,Mid-Market,337
2,Enterprise,144
3,NaN,1


In [21]:
from pathlib import Path

for p in Path("/content").rglob("customers.csv"):
    print("Found customers.csv at:", p)

Found customers.csv at: /content/New-Skills/customer-payments-analytics/data/raw/customers.csv


Find your project files

In [22]:
import pandas as pd
from pathlib import Path

customer_file = Path("/content/New-Skills/customer-payments-analytics/data/raw/customers.csv")
customers = pd.read_csv(customer_file)

print("Dataset shape:", customers.shape)
display(customers.head())

Dataset shape: (1001, 6)


,customer_id,customer_name,segment,region,customer_since,payment_terms_days
0,C00001,Customer 0001,Small Business,DACH,2020-04-04,30
1,C00002,Customer 0002,Small Business,DACH,2021-05-16,14
2,C00003,Customer 0003,Small Business,DACH,2021-06-22,30
3,C00004,Customer 0004,Small Business,DACH,2021-09-13,30
4,C00005,Customer 0005,Small Business,DACH,2024-07-13,30


Load the dataset using the correct path

In [23]:
duplicate_rows = customers[
    customers.duplicated(subset="customer_id", keep=False)
].sort_values("customer_id")

display(duplicate_rows)

,customer_id,customer_name,segment,region,customer_since,payment_terms_days
17,C00018,Customer 0018,Enterprise,DACH,2019-02-23,30
1000,C00018,Customer 0018,Enterprise,DACH,2019-02-23,30


investigate the duplicate before deciding whether to remove or merge anything.

In [24]:
customers_clean = customers.drop_duplicates().copy()

print("Original rows:", len(customers))
print("Cleaned rows:", len(customers_clean))
print("Rows removed:", len(customers) - len(customers_clean))
print("Duplicates remaining:", customers_clean.duplicated().sum())

Original rows: 1001
Cleaned rows: 1000
Rows removed: 1
Duplicates remaining: 0


Remove the exact duplicate

In [25]:
customers[customers["segment"].isna()]

,customer_id,customer_name,segment,region,customer_since,payment_terms_days
41,C00042,Customer 0042,NaN,DACH,2018-07-19,45


identified a missing value in the segment column

In [26]:
# Display the record with a missing segment
missing_segment = customers_clean[
    customers_clean["segment"].isna()
]

display(missing_segment)

# Count customers in each segment
print("\nCustomer counts by segment:")
print(customers_clean["segment"].value_counts(dropna=False))

,customer_id,customer_name,segment,region,customer_since,payment_terms_days
41,C00042,Customer 0042,NaN,DACH,2018-07-19,45



Customer counts by segment:
segment
Small Business    519
Mid-Market        337
Enterprise        143
NaN                 1
Name: count, dtype: int64


### Check payment terms distribution within each segment

In [27]:
# Group by segment and look at unique payment terms and counts
payment_terms_by_segment = customers_clean.groupby('segment')['payment_terms_days'].value_counts().reset_index(name='count')
display(payment_terms_by_segment)

,segment,payment_terms_days,count
0,Enterprise,30,79
1,Enterprise,45,31
2,Enterprise,14,18
3,Enterprise,60,15
4,Mid-Market,30,203
5,Mid-Market,45,62
6,Mid-Market,60,46
7,Mid-Market,14,26
8,Small Business,30,298
9,Small Business,45,104


In [28]:
print("Missing values by column:")
print(customers_clean.isna().sum())

Missing values by column:
customer_id           0
customer_name         0
segment               1
region                0
customer_since        0
payment_terms_days    0
dtype: int64


Investigate all missing values

In [33]:
# Inspect distinct region values
print("Unique region values:")
print(customers_clean["region"].unique())

# Count each value, including missing values
print("\nRegion counts:")
print(customers_clean["region"].value_counts(dropna=False))

Unique region values:
['DACH' 'Western Europe' 'Nordics' 'Southern Europe']

Region counts:
region
DACH               462
Western Europe     233
Nordics            171
Southern Europe    134
Name: count, dtype: int64


**Check inconsistent values:**

A dataset can have no missing values in a column but still contain inconsistent text. For example, DACH, dach, and DACH  could be interpreted as different values by a computer.

In [30]:
# Find region values with leading or trailing whitespace
region_values = customers_clean["region"]

inconsistent_regions = region_values[
    region_values.ne(region_values.str.strip())
]

print("Rows with extra spaces:", len(inconsistent_regions))
display(customers_clean.loc[inconsistent_regions.index])

Rows with extra spaces: 1


,customer_id,customer_name,segment,region,customer_since,payment_terms_days
76,C00077,Customer 0077,Small Business,DACH,2018-12-14,60


For example:
- 'DACH' — correct format
- 'DACH ' — extra space at the end
Even though these look almost identical, they can cause inaccurate grouping, filtering, and reporting.

In [31]:
customers_clean["region"] = (
    customers_clean["region"].str.strip()
)

print("Region values after cleaning:")
print(sorted(customers_clean["region"].dropna().unique()))

Region values after cleaning:
['DACH', 'Nordics', 'Southern Europe', 'Western Europe']


Correct the formatting safely. This removes leading and trailing whitespace without changing the actual region names.

In [32]:
print(
    "Rows with remaining leading/trailing spaces:",
    (
        customers_clean["region"]
        != customers_clean["region"].str.strip()
    ).sum()
)

Rows with remaining leading/trailing spaces: 0


Validate the result

In [34]:
print("Region values after cleaning:")
print(sorted(customers_clean["region"].dropna().unique()))

print(
    "\nRows with leading/trailing spaces:",
    (
        customers_clean["region"]
        != customers_clean["region"].str.strip()
    ).sum()
)

Region values after cleaning:
['DACH', 'Nordics', 'Southern Europe', 'Western Europe']

Rows with leading/trailing spaces: 0


In [35]:
print("Duplicate customer IDs:",
      customers_clean["customer_id"].duplicated().sum())

print("Invalid payment terms:",
      (customers_clean["payment_terms_days"] <= 0).sum())

print("Missing customer IDs:",
      customers_clean["customer_id"].isna().sum())

Duplicate customer IDs: 0
Invalid payment terms: 0
Missing customer IDs: 0


Check invalid values

In [36]:
invoices = pd.read_csv("/content/New-Skills/customer-payments-analytics/data/raw/invoices.csv")

print("Invoice dataset shape:", invoices.shape)

print("\nMissing values:")
print(invoices.isna().sum())

print("\nDuplicate invoice IDs:")
print(invoices["invoice_id"].duplicated().sum())

print("\nNon-positive invoice amounts:")
print((invoices["invoice_amount"] <= 0).sum())

Invoice dataset shape: (5000, 8)

Missing values:
invoice_id          0
customer_id         0
invoice_date        0
due_date            0
invoice_amount      0
currency            0
invoice_category    0
source_status       0
dtype: int64

Duplicate invoice IDs:
0

Non-positive invoice amounts:
1


Check invoice data:
- move from customer data to the invoice dataset.

In [37]:
invalid_invoices = invoices[
    invoices["invoice_amount"] <= 0
]

display(invalid_invoices)

,invoice_id,customer_id,invoice_date,due_date,invoice_amount,currency,invoice_category,source_status
122,INV000123,C00867,2025-02-26,2025-03-28,-125.0,EUR,Hardware,Paid


Find the problematic invoice